In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("../data/city_day_features.csv", parse_dates=["Date"])

# Keep rows where both today's AQI and yesterday's AQI exist
data = df.dropna(subset=["AQI", "AQI_Yesterday"])

# Time-based split
train = data[data["Year"] <= 2019]
test = data[data["Year"] == 2020]
print("Train rows:", len(train), "Test rows:", len(test))

# Baseline: predict today's AQI = yesterday's AQI
baseline_mae = mean_absolute_error(test["AQI"], test["AQI_Yesterday"])
print("Baseline MAE:", round(baseline_mae, 1))

Train rows: 18864 Test rows: 4262
Baseline MAE: 16.9


In [3]:
from sklearn.linear_model import LinearRegression

# Features we are allowed to use (all known BEFORE today)
features = ["AQI_Yesterday", "AQI_Trend", "PM25_7day_avg", "DayOfYear", "IsWeekend"]

# Linear Regression can't handle missing values, so keep complete rows only
train_lr = train.dropna(subset=features)
test_lr = test.dropna(subset=features)
print("Train rows:", len(train_lr), "Test rows:", len(test_lr))

# Train the model on the past
model = LinearRegression()
model.fit(train_lr[features], train_lr["AQI"])

# Predict 2020
predictions = model.predict(test_lr[features])
model_mae = mean_absolute_error(test_lr["AQI"], predictions)
print("Linear Regression MAE:", round(model_mae, 1))

# Fair comparison: baseline on the SAME rows
baseline_same = mean_absolute_error(test_lr["AQI"], test_lr["AQI_Yesterday"])
print("Baseline MAE (same rows):", round(baseline_same, 1))

Train rows: 18045 Test rows: 4175
Linear Regression MAE: 17.9
Baseline MAE (same rows): 16.9


In [4]:
from sklearn.ensemble import RandomForestRegressor

# Add City as numbers (one-hot encoding: one 0/1 column per city)
train_rf = pd.get_dummies(train_lr[features + ["City"]], columns=["City"])
test_rf = pd.get_dummies(test_lr[features + ["City"]], columns=["City"])

# Make sure test has the same columns as train
test_rf = test_rf.reindex(columns=train_rf.columns, fill_value=0)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(train_rf, train_lr["AQI"])

rf_pred = rf.predict(test_rf)
print("Random Forest MAE:", round(mean_absolute_error(test_lr["AQI"], rf_pred), 1))

Random Forest MAE: 19.0


In [5]:
# Target = change from yesterday
train_lr = train_lr.copy()
test_lr = test_lr.copy()
train_lr["AQI_Change"] = train_lr["AQI"] - train_lr["AQI_Yesterday"]

# Linear Regression on the change
model_change = LinearRegression()
model_change.fit(train_lr[features], train_lr["AQI_Change"])

# Predicted AQI = yesterday's AQI + predicted change
pred_change = test_lr["AQI_Yesterday"] + model_change.predict(test_lr[features])
print("Linear Regression (predicting change) MAE:",
      round(mean_absolute_error(test_lr["AQI"], pred_change), 1))
print("Baseline MAE:", round(mean_absolute_error(test_lr["AQI"], test_lr["AQI_Yesterday"]), 1))

Linear Regression (predicting change) MAE: 17.9
Baseline MAE: 16.9


In [6]:
train2 = data[data["Year"] <= 2018].dropna(subset=features)
test2 = data[data["Year"] == 2019].dropna(subset=features)

m = LinearRegression().fit(train2[features], train2["AQI"])
print("2019 test, Linear Regression MAE:",
      round(mean_absolute_error(test2["AQI"], m.predict(test2[features])), 1))
print("2019 test, Baseline MAE:",
      round(mean_absolute_error(test2["AQI"], test2["AQI_Yesterday"]), 1))

2019 test, Linear Regression MAE: 24.3
2019 test, Baseline MAE: 24.0


## Model results

**Test on 2020 (train 2015-2019):**

- Baseline (today = yesterday): MAE 16.9
- Linear Regression: MAE 17.9
- Linear Regression (predicting change): MAE 17.9
- Random Forest: MAE 19.0

**Test on 2019 (train 2015-2018):**

- Baseline: MAE 24.0
- Linear Regression: MAE 24.3

**Conclusion:** No model beat the baseline. A likely reason is that yesterday's AQI already explains most of today's AQI (correlation 0.91), so the other features add very little new information. Air quality changes slowly from day to day, which makes "today = yesterday" a hard guess to beat. This is a likely explanation, not a proven one.